In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

low_frequency_path = OUT_DIR / "handled_low_frequency_categories.csv"

# Let pandas detect the delimiter
data = pd.read_csv(low_frequency_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data



In [ ]:
data.columns

In [ ]:
data.info()

# Test some concepts in data before imputation

In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Before performing cohort-aware imputations, this cell checks the distribution
# of student start years using the column `sdo5_degree_COLLEGEJAAR`.
#
# Why this matters:
#   • Confirms that each cohort has enough observations to compute a reliable
#     cohort-specific median for `sdo1_previous_Final_Exam_Date`.
#   • Helps identify sparse or irregular cohorts that may need a fallback
#     (e.g., global median) during date imputation.
#   • Provides counts, percentages, and a visual bar chart for quick inspection.
# =============================================================================

year_col = "sdo5_degree_COLLEGEJAAR"

# Ensure numeric (just in case)
data[year_col] = pd.to_numeric(data[year_col], errors="coerce")

# Print counts
print("\n--- Cohort Year Counts ---")
print(data[year_col].value_counts(dropna=False).sort_index())

# Print percentages
print("\n--- Cohort Year Percentages ---")
print((data[year_col].value_counts(normalize=True, dropna=False)
       .sort_index() * 100).round(2))

# Optional bar chart
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
data[year_col].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution of Student Start Years (Cohorts)")
plt.xlabel("Cohort Year")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

Results: 

Looks perfect to do year-wise imputation.

In [ ]:
# NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

# Missing Values Imputation Strategy

In [ ]:
# -----------------------------------------------------------------------------
# Missing Value Imputation Plan — Updated with Cohort-Aware Strategy
# -----------------------------------------------------------------------------
# Dataset size:
#   N = 47,946 rows
#
# Missingness Summary (by column group)
#
# Orientation participation (≈81% missing — structural):
#   • sdo2_orientation_First_Event_Date ................. 80.74%
#   • sdo2_orientation_Last_Event_Date .................. 80.74%
#   • sdo2_orientation_Number_of_Events_Attended ........ 80.74%
#   • sdo2_orientation_STUDENTNUMMER .................... 80.74%
#   • sdo2_orientation_Number_of_Event_Types ............ 80.74%
#
# Academic performance (≈9–24% missing):
#   • Averages (A/B): missing because exam not taken.
#   • Percentages/Potentials: missing because no exam activity occurred.
#
# SKC and event-related dates:
#   • sdo2_skc_SKC_DATUM ................................ 10.94%
#
# Distances (≈3–9% missing):
#   • prev_distance_*  .................................. 9.46%
#   • postal_distance_* .................................. 3.20%
#
# Demographics:
#   • age_start_study .................................... 7.25%
#   • Dutch_nationality .................................. 7.25%
#
# Previous education:
#   • Final exam date (sdo1_previous_Final_Exam_Date) .... 1.62%
#
# Degree-level and has_* flags:
#   • Almost complete; no imputation needed.
#
# -----------------------------------------------------------------------------
# Imputation Plan
# -----------------------------------------------------------------------------
#
# 1) Orientation participation (structural missingness):
#      • Keep event dates as NaT (students who never attended).
#      • Numeric counts → fill with 0.
#      • Add <col>_missing_flag for interpretability.
#
# 2) Academic performance:
#      A) Average_Grade_A / Average_Grade_B:
#         • Create <col>_Sat_Exam flag: 1 = exam sat, 0 = missing.
#         • Impute missing grades using the median among students who sat.
#         • Do NOT create *_missing_flag for A/B averages.
#
#      B) Percentage_* and Potential_*:
#         • Missing means “no activity”.
#         • Fill with 0, and add <col>_missing_flag.
#
# 3) Distances (random-like missingness):
#      • Median imputation for all distance columns.
#      • Add <col>_missing_flag.
#
# 4) Demographics:
#      • Age → median (+ missing_flag)
#      • Dutch nationality → mode (+ missing_flag)
#
# 5) Dates with partial missingness:
#
#      A) sdo1_previous_Final_Exam_Date:
#           • Convert to datetime.
#           • Impute using **cohort-wise median** based on:
#                 sdo5_degree_COLLEGEJAAR
#           • Apply **global median fallback** if a cohort
#             has no valid exam dates.
#           • Add <col>_missing_flag before imputation.
#
#      B) sdo2_skc_SKC_DATUM:
#           • Keep NaT — structural (“no SKC event recorded”).
#           • Add <col>_missing_flag.
#
#      C) Orientation event dates:
#           • Keep NaT — structural.
#           • Add <col>_missing_flag.
#
# 6) Columns with no missing values:
#      • No action required.
#
# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------
# • Structural missingness preserved (NaT or 0).
# • Academic grades: Sat_Exam flags + median imputation for sitters.
# • Numeric gaps filled with median; categorical with mode.
# • Previous final exam date now uses **cohort-aware imputation**
#     → median per sdo5_degree_COLLEGEJAAR cohort
#     → fallback to global median when needed.
# • All imputations maintain interpretability and model readiness.
# -----------------------------------------------------------------------------

In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Function: impute_all(data)
#
# Structured missing-value imputation for the student dataset (N = 47,946),
# preserving structural missingness (e.g., no orientation activity, no exams).
#
# Key imputations (per verified plan):
#   1) Orientation (~81% missing, structural):
#        - Numeric counts → fill with 0.
#        - Dates → keep as NaT (no imputation).
#   2) Academic performance:
#        - Average grades (A, B): create Sat_Exam flags
#            * <grade>_Sat_Exam = 1 if grade present (sat), 0 if missing.
#            * Impute missing grades with the median among sitters only.
#        - Other academic metrics (Percentage_*, Potential_*):
#            * Fill with 0 and add *_missing_flag.
#   3) Distances (≈3–9% missing):
#        - Numeric distances → median (+ *_missing_flag).
#   4) Demographics:
#        - Age → median (+ flag)
#        - Dutch nationality → mode (+ flag)
#   5) Dates with partial missingness:
#        - sdo2_skc_SKC_DATUM → keep NaT (structural “no event”)
#        - sdo1_previous_Final_Exam_Date:
#              * Impute using **cohort-wise median** based on sdo5_degree_COLLEGEJAAR
#              * Apply **global median fallback** if a cohort lacks valid dates
#
# Flags:
#   For every imputed or structurally-missing column → <column>_missing_flag.
#   EXCEPTION: For Average Grade A/B, use <grade>_Sat_Exam instead.
#
# Output:
#   Returns a new DataFrame with imputations applied and all indicator flags added.
# =============================================================================



def impute_all(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # -----------------------------
    # 1) Define column groups
    # -----------------------------
    # Orientation: numeric counts (set to 0 if missing)
    orientation_count_cols = [
        "sdo2_orientation_Number_of_Event_Types",
        "sdo2_orientation_Number_of_Events_Attended",
        "sdo2_orientation_STUDENTNUMMER",
    ]
    # Orientation: dates (structural NaT)
    orientation_date_cols_structural = [
        "sdo2_orientation_First_Event_Date",
        "sdo2_orientation_Last_Event_Date",
    ]

    # Dates with partial missingness
    date_keep_nat = ["sdo2_skc_SKC_DATUM"]  # Keep as NaT
    date_cols_median = ["sdo1_previous_Final_Exam_Date"]

    # Cohort column for cohort-wise date imputation
    cohort_col = "sdo5_degree_COLLEGEJAAR"

    # Academic performance groups
    academic_grades = [
        "sdo6_results_Average_Grade_A",
        "sdo6_results_Average_Grade_B",
    ]
    academic_other_zero = [
        "sdo6_results_Percentage_Credits_B",
        "sdo6_results_Potential_Credits_B",
        "sdo6_results_Percentage_Credits_A",
        "sdo6_results_Potential_Credits_A",
    ]

    # Distances → median (removed *_postcode columns)
    distance_numeric = [
        "sdo1_prev_distance_distance_to_3584_CS",
        "sdo1_prev_distance_distance_to_3812_PA",
        "sdo1_postal_distance_distance_to_3584_CS",
        "sdo1_postal_distance_distance_to_3812_PA",
    ]

    # Demographics
    demo_age_median = ["sdo1_characteristics_age_start_study"]
    demo_nat_mode   = ["sdo1_characteristics_Dutch_nationality"]

    all_date_cols = list(
        dict.fromkeys(
            orientation_date_cols_structural + date_keep_nat + date_cols_median
        )
    )

    # -----------------------------
    # 2) Helpers
    # -----------------------------
    def add_flag(col: str):
        flag = f"{col}_missing_flag"
        df[flag] = df[col].isna().astype("int8")

    def to_datetime_cols(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")

    def median_datetime(series: pd.Series) -> pd.Timestamp:
        return series.median()

    def mode_safe(series: pd.Series):
        m = series.mode(dropna=True)
        return m.iloc[0] if len(m) > 0 else np.nan

    def coerce_numeric(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    # -----------------------------
    # 3) Convert datatypes
    # -----------------------------
    to_datetime_cols(all_date_cols)
    coerce_numeric(
        orientation_count_cols
        + academic_grades
        + academic_other_zero
        + distance_numeric
        + demo_age_median
        + [cohort_col]  # ensure cohort is numeric if present
    )

    # -----------------------------
    # 4) Orientation (structural)
    # -----------------------------
    for c in orientation_count_cols:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    for c in orientation_date_cols_structural:
        if c in df.columns:
            add_flag(c)  # keep NaT (no imputation)

    # -----------------------------
    # 5) Dates
    # -----------------------------
    # Keep NaT for SKC date (structural missingness)
    for c in date_keep_nat:
        if c in df.columns:
            add_flag(c)  # Keep NaT as “no event recorded”

    # Impute previous exam date:
    #   - cohort-wise median based on sdo5_degree_COLLEGEJAAR
    #   - fallback to global median if needed
    for c in date_cols_median:
        if c in df.columns:
            add_flag(c)

            # Cohort-wise median if cohort column is available
            if cohort_col in df.columns:
                # compute median exam date within each cohort and align back to rows
                cohort_medians = df.groupby(cohort_col)[c].transform("median")
                df[c] = df[c].fillna(cohort_medians)

            # Fallback: global median for any rows still NaT
            if df[c].isna().any():
                med_global = median_datetime(df[c])
                if not pd.isna(med_global):
                    df[c] = df[c].fillna(med_global)

    # -----------------------------
    # 6) Academic performance
    # -----------------------------
    # 6a) Average grades → Sat_Exam flags + impute with median among sitters
    for gcol in academic_grades:
        if gcol in df.columns:
            sat_flag_col = f"{gcol}_Sat_Exam"
            df[sat_flag_col] = df[gcol].notna().astype("int8")
            med_sat = df.loc[df[gcol].notna(), gcol].median(skipna=True)
            if not pd.isna(med_sat):
                df[gcol] = df[gcol].fillna(med_sat)

    # 6b) Other academic metrics (reflecting no-exam activity) → ZERO (+ flags)
    for c in academic_other_zero:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    # -----------------------------
    # 7) Distances → median
    # -----------------------------
    for c in distance_numeric:
        if c in df.columns:
            add_flag(c)
            med = df[c].median(skipna=True)
            df[c] = df[c].fillna(med)

    # -----------------------------
    # 8) Demographics
    # -----------------------------
    # Age → median
    for c in demo_age_median:
        if c in df.columns:
            add_flag(c)
            med = df[c].median(skipna=True)
            df[c] = df[c].fillna(med)

    # Dutch nationality → mode
    for c in demo_nat_mode:
        if c in df.columns:
            add_flag(c)
            m = mode_safe(df[c])
            df[c] = df[c].fillna(m)

    # -----------------------------
    # 9) Clean-up for numeric consistency
    # -----------------------------
    if "sdo2_orientation_STUDENTNUMMER" in df.columns:
        df["sdo2_orientation_STUDENTNUMMER"] = df["sdo2_orientation_STUDENTNUMMER"].astype("Int64")

    return df

# Example usage:
data = impute_all(data)

In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(nan_sums)

In [ ]:
#del data ['sdo2_orientation_Last_Event_Date']
#del data ['sdo2_orientation_Last_Event_Date']

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
print(data['sdo5_degree_POSTAL_COUNTRY_NL'].replace('', np.nan).dropna().value_counts())

In [ ]:
data.to_csv(f"{OUT_DIR}/missing_values_imputed.csv", index=False)
print(f"✅ Saved to: {OUT_DIR}/missing_values_imputed.csv")